# Week 3, Notebook 4: Time Series Forecasting

**Guessing tomorrow, and grading your guess honestly.**

*Author: The Genius Project Year 3*

---

### What you will build
- Four ways to forecast tomorrow's price, from dead simple to clever.
- A fair test that hides the future and grades each method.
- The habit every honest analyst has: **always compare against a dumb baseline.**

### The big idea
Forecasting is not about being a fortune teller. It is about making a guess, checking
it against what really happened, and keeping the method that is least wrong. The
surprise of this notebook is how hard the *simplest* guess is to beat.

### Step 1: Split time into "past" and "future"

You must never grade a forecast on data it was allowed to see. So we cut the history
in two: an early **training** part the methods can study, and a recent **test** part
we keep hidden and use only for grading. Because this is time data, the split is by
date, never shuffle a time series.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

stock = pd.read_csv("data/daily_stock.csv", parse_dates=["date"]).set_index("date")
price = stock["price"]

TEST_DAYS = 60                       # grade on the final 60 trading days
train = price.iloc[:-TEST_DAYS]
test = price.iloc[-TEST_DAYS:]
print(f"Training on {len(train)} days, testing on the last {len(test)} days.")


def mae(actual, predicted):
    "Mean Absolute Error: on average, how many dollars off were we?"
    return np.mean(np.abs(np.array(actual) - np.array(predicted)))


def rmse(actual, predicted):
    "Root Mean Squared Error: like MAE but punishes big misses harder."
    return np.sqrt(np.mean((np.array(actual) - np.array(predicted)) ** 2))

### Step 2: The baseline every pro starts with, "tomorrow = today"

The **naive forecast** says the best guess for tomorrow's price is today's price.
It sounds too simple, but it is hard to beat, because a price barely moves in a day. If a fancy method cannot beat this, the fancy method is worthless.

In [ ]:
# For each test day, guess the price from the day before.
naive_pred = price.shift(1).iloc[-TEST_DAYS:]

naive_mae = mae(test, naive_pred)
print(f"Naive forecast  ->  MAE ${naive_mae:.2f}  |  RMSE ${rmse(test, naive_pred):.2f}")
print("Read as: on a typical day, this guess was off by about "
      f"${naive_mae:.2f}.")

### Step 3: A moving average

Maybe smoothing helps: guess tomorrow as the average of the last few days. This calms
down day-to-day noise, but it also lags behind when the price is trending,
a trade-off you will see in the error.

In [ ]:
WINDOW = 5
ma_pred = price.rolling(WINDOW).mean().shift(1).iloc[-TEST_DAYS:]

print(f"{WINDOW}-day moving average  ->  MAE ${mae(test, ma_pred):.2f}  "
      f"|  RMSE ${rmse(test, ma_pred):.2f}")

### Step 4: Exponential smoothing

A moving average treats the last 5 days as equally important. **Exponential
smoothing** is smarter: it weights yesterday most, the day before less, and so on,
fading into the past. The knob `alpha` (between 0 and 1) sets how fast the memory
fades. Pandas has it built in as `ewm`.

In [ ]:
ALPHA = 0.6
ewm_pred = price.ewm(alpha=ALPHA).mean().shift(1).iloc[-TEST_DAYS:]

print(f"Exponential smoothing (alpha={ALPHA})  ->  MAE ${mae(test, ewm_pred):.2f}  "
      f"|  RMSE ${rmse(test, ewm_pred):.2f}")

### Step 5: A simple learned model (autoregression)

Our first *model*: predict tomorrow from the last few days using a straight-line fit
learned from the training data. Feeding a value's own past back in is called
**autoregression**. We build "lag" columns (yesterday, two days ago, ...) as the
inputs and let `LinearRegression` find the best weights.

In [ ]:
from sklearn.linear_model import LinearRegression

LAGS = 5
# Build a table: each row = the last 5 prices, and the price the next day.
data = pd.DataFrame({"y": price})
for k in range(1, LAGS + 1):
    data[f"lag{k}"] = price.shift(k)
data = data.dropna()

feature_cols = [f"lag{k}" for k in range(1, LAGS + 1)]
X, y = data[feature_cols], data["y"]

X_train, y_train = X.iloc[:-TEST_DAYS], y.iloc[:-TEST_DAYS]
X_test, y_test = X.iloc[-TEST_DAYS:], y.iloc[-TEST_DAYS:]

model = LinearRegression().fit(X_train, y_train)
ar_pred = model.predict(X_test)

print(f"Autoregression (last {LAGS} days)  ->  MAE ${mae(y_test, ar_pred):.2f}  "
      f"|  RMSE ${rmse(y_test, ar_pred):.2f}")

### Step 6: Put every method on the same chart

Numbers are convincing; a picture is memorable. We plot the real prices for the test
window against each forecast, then rank the methods by error.

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(test.index, test.values, color="black", linewidth=2, label="what really happened")
plt.plot(test.index, naive_pred.values, "--", label="naive")
plt.plot(test.index, ma_pred.values, "--", label=f"{WINDOW}-day average")
plt.plot(test.index, ewm_pred.values, "--", label="exp. smoothing")
plt.plot(y_test.index, ar_pred, "--", label="autoregression")
plt.title("Forecasts vs reality on the hidden test days")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.show()

scores = {
    "naive": mae(test, naive_pred),
    f"{WINDOW}-day average": mae(test, ma_pred),
    "exp. smoothing": mae(test, ewm_pred),
    "autoregression": mae(y_test, ar_pred),
}
print("Ranking by error (lower is better):")
for name, score in sorted(scores.items(), key=lambda kv: kv[1]):
    print(f"  {name:<16} MAE ${score:.2f}")

### The honest takeaway

Look how close the fancy methods land to the humble naive guess. That is not a bug
,  it is the deep lesson of Notebook 3 showing up again: because returns have
almost no memory, the best guess for tomorrow's price really is close to today's. A
forecast is only impressive when it clearly beats the naive baseline, so **always
print the baseline first.**

### What you learned
- Split a time series by **date**, never by shuffling.
- **MAE** and **RMSE** turn "good guess" into a gradeable number.
- The **naive** forecast is a very strong baseline.
- Moving averages, exponential smoothing, and autoregression are the classic toolkit.

### Try it yourself
- Change `TEST_DAYS` to `120`. Does the ranking of methods change?
- Try `ALPHA = 0.2` in the smoothing. Smoother, or more lagging?
- Forecast the budget's monthly `spending` instead. (Hint: load `monthly_budget.csv`
  and reuse the naive and moving-average code.)

### Next
Every method so far used only the price's own past. Notebook 5 brings in **machine
learning**, which can mix many clues at once, and asks the sharp question: can
it predict whether the stock goes *up or down* tomorrow?